In [ ]:
# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List
import seaborn as sns
import re
from collections import defaultdict

In [ ]:
# Helper maps / constants

region_mapping = {"0": "FRA", "1": "SF", "2": "BAN", "3": "SYD"}


# Plotting
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

output_base_path = Path("output")

detailed_output_dir = Path("output/detailed")
detailed_output_dir.mkdir(exist_ok=True)

comparison_output_dir = Path("output/comparison")
comparison_output_dir.mkdir(exist_ok=True)

ycsb_output_dir = Path("output/ycsb")
ycsb_output_dir.mkdir(exist_ok=True)

kvstore_output_dir = Path("output/kvstore")
kvstore_output_dir.mkdir(exist_ok=True)

bar_linewidth = 1

single_figsize = (6, 4)
double_figsize = (8, 5)

title_fontsize = 14
xylabel_fontsize = 12
legend_fontsize = 10

In [ ]:
N = 1000  # number of items (ranks)
alpha = 1.1  # Zipf exponent

# Generate Zipfian-like data
ranks = np.arange(1, N + 1)
freqs = 1 / ranks**alpha
freqs /= freqs.sum()  # normalize

# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=double_figsize)

axes[0].plot(ranks, freqs, color="royalblue")
axes[0].set_title("Zipfian Distribution (Linear Scale)")
axes[0].set_xlabel("Rank", fontsize=xylabel_fontsize)
axes[0].set_ylabel("Frequency (normalized)", fontsize=xylabel_fontsize)
axes[0].grid(True, linestyle="--", linewidth=0.5)

axes[1].loglog(
    ranks, freqs, marker="o", markersize=3, linestyle="none", color="darkorange"
)
axes[1].set_title("Zipfian Distribution (Log–Log Scale)")
axes[1].set_xlabel("Rank (log)", fontsize=xylabel_fontsize)
axes[1].set_ylabel("Frequency (log)", fontsize=xylabel_fontsize)
axes[1].grid(True, which="both", linestyle="--", linewidth=0.5)

plt.tight_layout()
plt.show()

# Evaluation

## Directory structure

- Base directory: data
- Each benchmark run is stored in a separate directory and has the prefix `ycsb_`
  - All benchmark files in the directory have the prefix `ycsb_`, then `l_` if
  it was the load phase, or `t_` if it was the transaction phase.
  - After that, the benchmark name is added. By convention, it is 
    `{user defined name}-{transaction count}-{run number}`. For example,
    `trivial-3000-4` is the 4th run of the `trivial` configuration of ISOS, with
    3000 operations per client replica, i.e., 12000 operations total with 4
    client nodes.
  - After that, the client node ID is appended as `_{client id}`, ranging from
    `0`-`3` for the transaction nodes. The client for the load phase has the ID `5`.
  - Each client saves a `.csv` and `.raw` file. The CSV file is the default YCSB
    human-readable overview. As we do the evaluation separately in Python, we
    need the latency from each request, as the `timeseries` or `histogram` are
    not as detailed / granular. The `raw` file is formatted as CSV.
- Example files:
  - Benchmark name `trivial-raw-3000-2`, i.e., the second run of trivial-raw
  with 3000 operations. Files from the YCSB transaction phase on client node 3:
    - CSV file: `data/ycsb_trivial-raw-3000-2/ycsb_t_trivial-raw-3000-2_3.csv`
    - RAW file: `data/ycsb_trivial-raw-3000-2/ycsb_t_trivial-raw-3000-2_3.raw`
    - For analysis of all runs of all clients, add `trivial-raw-3000` to the
    `ycsb_raw_benchmarks` array
  
## Evaluation Details

- If a benchmark has multiple runs, all runs are considered for the evaluation.
- Only the `.raw` files are evaluated. The `.csv` files are for 
- `READ` and `UPDATE` commands are combined, `CLEANUP` commands are ignored
- We generate the following graphs for each benchmark:
  - Overview of performance: Average overall client latency (ms) and throughput
    (ops/s) across all benchmark runs and all clients for the given benchmark
  - Latency percentiles (p1, p5, p50, p90, p95, p99, p99.9) overall, with
    percentile bands
  - Latency percentiles split by client
  - Average overall latency over (benchmark) time, in 1 second buckets


## Data Loading

In [ ]:
# Functions to load raw data to dataframe


def find_ycsb_benchmark_files(
    data_dir: str,
    benchmark_name: str,
    dir_prefix: str,
    file_prefix: str,
) -> Dict[int, List[Path]]:
    """
    Find all raw files for a benchmark pattern across all runs and clients.

    Returns: {run_number: [client_0_path, client_1_path, ...]}
    """

    # First, get directories of all runs
    # Pattern: ycsb_{benchmark_pattern}-{run_number}
    base_data_dir = Path(data_dir)
    runs = defaultdict(list)
    run_paths = base_data_dir.rglob(f"**/{dir_prefix}{benchmark_name}-*")
    for run_dir in run_paths:
        # Extract run number from directory name
        match = re.search(rf"{benchmark_name}-(\d+)$", run_dir.name)
        if not match:
            continue

        run_num = int(match.group(1))

        # Find all client files (0-3) for this run

        for client_id in range(4):
            pattern = f"{file_prefix}{benchmark_name}-*_{client_id}.raw"
            client_files = list(run_dir.glob(pattern))

            if client_files:
                runs[run_num].append((client_id, client_files[0]))

    return dict(runs)


def parse_raw_file(filepath: Path, client_id: int, run_number: int) -> pd.DataFrame:
    """Parse a YCSB raw file and add metadata"""
    data = []

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if (
                not line
                or line.startswith("#")
                or "latency raw data" in line.lower()
                or line.startswith("op,")
            ):
                continue

            parts = line.split(",")
            if len(parts) == 3:
                op, timestamp, latency = parts
                data.append(
                    {
                        "operation": op,
                        "absolute_timestamp_ms": int(timestamp),
                        "latency_us": int(latency),
                        "client_id": client_id,
                        "run_number": run_number,
                    }
                )

    df = pd.DataFrame(data)
    if len(df) > 0:
        # Remove "CLEANUP" operations from dataframe
        df = df[df["operation"] != "CLEANUP"]

        # Convert data to more useful units
        # Latency in ms
        df["latency_ms"] = df["latency_us"] / 1000
        df = df.drop(columns=["latency_us"])
        # Relative timestamp of the operation in the replica, starting from the
        # first measurement of a run
        start_time = df["absolute_timestamp_ms"].min()
        df["rel_timestamp_ms"] = (df["absolute_timestamp_ms"] - start_time) / 1000

    return df


def load_benchmark_data(
    data_dir: str, benchmark_pattern: str, dir_prefix: str, file_prefix: str
) -> pd.DataFrame:
    """
    Load all data for a benchmark pattern across all runs and clients, and then
    do som

    Args:
        data_dir: Base data directory
        benchmark_pattern: Pattern like "trivial-raw-3000"

    Returns:
        Combined DataFrame with all runs and clients
    """
    runs = find_ycsb_benchmark_files(
        data_dir, benchmark_pattern, dir_prefix, file_prefix
    )

    if not runs:
        raise ValueError(f"No data found for benchmark pattern: {benchmark_pattern}")

    all_dfs = []

    for run_num, client_files in sorted(runs.items()):
        # print(f"Loading run {run_num}...")
        for client_id, filepath in sorted(client_files):
            # print(f"  - Client {client_id}: {filepath.name}")
            df = parse_raw_file(filepath, client_id, run_num)
            all_dfs.append(df)

    # Combine all data
    combined_df = pd.concat(all_dfs, ignore_index=True)

    # For each run, calculate the overall run number
    combined_df["overall_run_ms"] = combined_df.groupby("run_number")[
        "absolute_timestamp_ms"
    ].transform(lambda x: (x - x.min()))

    print(f"{benchmark_pattern}: Loaded {len(combined_df):,} operations")
    print(f"  - {combined_df['run_number'].nunique()} runs")
    # print(f"  - Avg. Latenyc: {combined_df['latency_ms'].average()} ms")

    return combined_df


def load_benchmark_data_into_dict(
    data_dir: str, benchmarks_dict: dict, dir_prefix: str, file_prefix: str
) -> dict:
    for key in benchmarks_dict:
        benchmarks_dict[key] = load_benchmark_data(
            data_dir, key, dir_prefix, file_prefix
        )
    return benchmarks_dict


def first_request_per_client(df: pd.DataFrame) -> pd.DataFrame:
    """Return the first observed request for each (run_number, client_id)."""

    # idx of the row with minimum rel_timestamp_ms per (run_number, client_id)
    idx = df.groupby(["run_number", "client_id"])["rel_timestamp_ms"].idxmin()
    # select useful columns and rename
    firsts = df.loc[idx, ["run_number", "client_id", "absolute_timestamp_ms"]].copy()

    # Compute per-run minimum absolute timestamp
    run_min_map = df.groupby("run_number")["absolute_timestamp_ms"].min().to_dict()
    firsts["diff_after_initial_ms"] = firsts["absolute_timestamp_ms"] - firsts[
        "run_number"
    ].map(run_min_map)

    firsts = firsts.sort_values(["run_number", "diff_after_initial_ms"]).reset_index(
        drop=True
    )
    return firsts


In [ ]:
# Get general summary data from dataframes


def filter_error_df(df: pd.DataFrame) -> pd.DataFrame:
    filtered_df = df[~df["operation"].isin(["READ-ERROR", "UPDATE-ERROR"])].copy()
    return filtered_df


def calculate_overall_statistics(df: pd.DataFrame) -> Dict:
    """Calculate statistics across all runs and clients"""

    df_noerr = filter_error_df(df)

    # Overall statistics
    total_ops = len(df)
    latencies = df_noerr["latency_ms"]

    # Count errored operations (operation names ending with "-ERROR")
    error_count = df["operation"].fillna("").str.endswith("-ERROR").sum()
    error_rate = error_count / total_ops if total_ops > 0 else 0.0

    # Per-run statistics for duration and throughput
    run_stats = []
    for run_num in df["run_number"].unique():
        run_df = df[df["run_number"] == run_num]
        duration_s = (
            run_df["absolute_timestamp_ms"].max()
            - run_df["absolute_timestamp_ms"].min()
        ) / 1000
        throughput = len(run_df) / duration_s if duration_s > 0 else 0
        run_stats.append(
            {
                "run": run_num,
                "duration_s": duration_s,
                "throughput": throughput,
                "operations": len(run_df),
            }
        )

    run_stats_df = pd.DataFrame(run_stats)

    # Percentiles
    percentiles = [1, 5, 50, 90, 95, 99, 99.9]

    stats = {
        "total_operations": total_ops,
        "total_runs": df["run_number"].nunique(),
        "total_clients": df["client_id"].nunique(),
        "avg_duration_s": run_stats_df["duration_s"].mean(),
        "avg_throughput_ops_s": run_stats_df["throughput"].mean(),
        "throughput_std": run_stats_df["throughput"].std(),
        "avg_latency_ms": latencies.mean(),
        "latency_std_ms": latencies.std(),
        "min_latency_ms": latencies.min(),
        "max_latency_ms": latencies.max(),
        "percentiles": {f"p{p}": np.percentile(latencies, p) for p in percentiles},
        "run_details": run_stats_df,
        "error_count": int(error_count),
        "error_rate": float(error_rate),
    }

    return stats


def summarize_benchmark_to_df(
    df: pd.DataFrame,
    benchmark_name: str,
    expected_runs: int = 15,
    clients_per_run: int = 4,
) -> pd.DataFrame:
    """Return a single-row DataFrame summarizing benchmark-level stats for cross-benchmark comparison."""
    stats = calculate_overall_statistics(df)

    row = {
        "benchmark": benchmark_name,
        "total_operations": stats.get("total_operations"),
        "total_runs": stats.get("total_runs"),
        "expected_runs": expected_runs,
        "missing_runs": max(0, expected_runs - int(stats.get("total_runs", 0))),
        "total_clients_observed": stats.get("total_clients"),
        "expected_clients_per_run": clients_per_run,
        "avg_duration_s": stats.get("avg_duration_s"),
        "avg_throughput_ops_s": stats.get("avg_throughput_ops_s"),
        "throughput_std": stats.get("throughput_std"),
        "avg_latency_ms": stats.get("avg_latency_ms"),
        "latency_std_ms": stats.get("latency_std_ms"),
        "min_latency_ms": stats.get("min_latency_ms"),
        "max_latency_ms": stats.get("max_latency_ms"),
        "error_count": stats.get("error_count"),
        "error_rate": stats.get("error_rate"),
    }

    # add percentile columns (e.g. perc_p1, perc_p5, ...)
    for pname, pval in stats.get("percentiles", {}).items():
        row[f"perc_{pname}"] = pval

    # add per-run operation stats if available
    run_df = stats.get("run_details")
    if isinstance(run_df, pd.DataFrame) and not run_df.empty:
        row["mean_ops_per_run"] = run_df["operations"].mean()
        row["median_ops_per_run"] = run_df["operations"].median()
        row["runs_with_ops"] = len(run_df)

    return pd.DataFrame([row])


def summary_from_df_dict(
    dfs_dict: dict,
    expected_runs: int,
) -> dict:
    result = {}
    for key in dfs_dict:
        result[key] = summarize_benchmark_to_df(dfs_dict[key], key, expected_runs, 4)

    return result


def summary_from_df_array_dict(
    dfs_dict: dict,
    expected_runs: int,
) -> dict:
    result = {}
    for key in dfs_dict:
        arr = dfs_dict[key]
        result[key] = [summarize_benchmark_to_df(x, key, expected_runs, 4) for x in arr]

    return result

In [ ]:
# Collect benchmark data from multiple runs into dictionaries

ycsb_dir_prefix = "ycsb_"
ycsb_file_prefix = "ycsb_t_"  # ignore load phase files

kvstore_dir_prefix = "kvstore_"
kvstore_file_prefix = "kvstore_"

# Constants

## Optimization benchmarks
NO_OPT_500 = "no-opt-500"
RECORD_CACHE_500 = "record-cache-500"
CMD_CACHE_500 = "cmd-cache-500"
HIGHEST_CONFLICT_500 = "highest-conflict-500"
DEPGRAPH_CACHED_500 = "depgraph-cached-500"
DEPGRAPH_CONCURRENT_TEST_500_LOCKED = "depgraph-concurrent-test-500"
DEPGRAPH_CONCURRENT_TEST1_500_PHASER = "depgraph-concurrent-test1-500"
CONCDEP_HIGHESTCONFLICT_500 = "concdep-highestconflict-500"
SEQ_EXEC_25K_500 = "seq-exec-25k-500"
CONC_EXEC_25K_500 = "conc-exec-25k-500"
CONC_EXEC_25K_50W_500 = "conc-exec-25k-50w-500"
BATCH_5_500 = "batch-5-500"
BATCH_5_25MS_500 = "batch-5-25ms-500"
BATCH_5_50MS_500 = "batch-5-50ms-500"
BATCH_5_150MS_500 = "batch-5-150ms-500"

## BFT Smart Baseline Measurements
YCSB_BFT_SMART_LEADER0_50R_50W_3000 = "bft-smart-leader0-50w-3000"
YCSB_BFT_SMART_LEADER1_50R_50W_3000 = "bft-smart-leader1-50w-3000"
YCSB_BFT_SMART_LEADER2_50R_50W_3000 = "bft-smart-leader2-50w-3000"
YCSB_BFT_SMART_LEADER3_50R_50W_3000 = "bft-smart-leader3-50w-3000"

YCSB_ISOS_50R_50W_B5 = "isos-optimized-50w-3000"
YCSB_ISOS_50R_50W_B25 = "isos-optimized-50w-b25-3000"

YCSB_BFT_SMART_LEADER0_100R_0W_3000 = "bft-smart-leader0-0w-3000"
YCSB_BFT_SMART_LEADER1_100R_0W_3000 = "bft-smart-leader1-0w-3000"
YCSB_BFT_SMART_LEADER2_100R_0W_3000 = "bft-smart-leader2-0w-3000"
YCSB_BFT_SMART_LEADER3_100R_0W_3000 = "bft-smart-leader3-0w-3000"

YCSB_ISOS_100R_0W_B5 = "isos-optimized-0w-3000"
YCSB_ISOS_100R_0W_B25 = "isos-optimized-0w-b25-3000"

YCSB_BFT_SMART_LEADER0_95R_5W_3000 = "bft-smart-leader0-3000"
YCSB_BFT_SMART_LEADER1_95R_5W_3000 = "bft-smart-leader1-3000"
YCSB_BFT_SMART_LEADER2_95R_5W_3000 = "bft-smart-leader2-3000"
YCSB_BFT_SMART_LEADER3_95R_5W_3000 = "bft-smart-leader3-3000"

YCSB_ISOS_95R_5W_B5 = "isos-optimized-3000"
YCSB_ISOS_95R_5W_B25 = "isos-optimized-b25-3000"

## KVStore

KV_BFT_SMART_LEADER0_1000 = "smart-leader0-1000"
KV_BFT_SMART_LEADER1_1000 = "smart-leader1-1000"
KV_BFT_SMART_LEADER2_1000 = "smart-leader2-1000"
KV_BFT_SMART_LEADER3_1000 = "smart-leader3-1000"

KV_ISOS_C0_1000 = "optimized-c0-1000"
KV_ISOS_C2_1000 = "optimized-c2-1000"
KV_ISOS_C5_1000 = "optimized-c5-1000"
KV_ISOS_C100_1000 = "optimized-c100-1000"

## ISOS Optimized Measurements

## Collect YCSB ISOS Optimization benchmarks

isos_opt_dfs = {
    # trivial
    NO_OPT_500: [],
    # record-cache
    RECORD_CACHE_500: [],
    # serialization
    CMD_CACHE_500: [],
    # conflicts
    HIGHEST_CONFLICT_500: [],
    # dep-graph
    DEPGRAPH_CACHED_500: [],
    DEPGRAPH_CONCURRENT_TEST_500_LOCKED: [],
    DEPGRAPH_CONCURRENT_TEST1_500_PHASER: [],
    CONCDEP_HIGHESTCONFLICT_500: [],
    # execution
    SEQ_EXEC_25K_500: [],
    CONC_EXEC_25K_500: [],
    CONC_EXEC_25K_50W_500: [],
    # batching
    BATCH_5_500: [],  # 100ms
    BATCH_5_25MS_500: [],
    BATCH_5_50MS_500: [],
    BATCH_5_150MS_500: [],
}

load_benchmark_data_into_dict(
    "data/isos-opt", isos_opt_dfs, ycsb_dir_prefix, ycsb_file_prefix
)

## 50r_50w benchmarks

ycsb_smart_50r_50w_dfs = {
    YCSB_BFT_SMART_LEADER0_50R_50W_3000: [],
    YCSB_BFT_SMART_LEADER1_50R_50W_3000: [],
    YCSB_BFT_SMART_LEADER2_50R_50W_3000: [],
    YCSB_BFT_SMART_LEADER3_50R_50W_3000: [],
}

load_benchmark_data_into_dict(
    "data/ycsb/50r_50w/bft-smart",
    ycsb_smart_50r_50w_dfs,
    ycsb_dir_prefix,
    ycsb_file_prefix,
)

ycsb_isos_50r_50w_dfs = {
    YCSB_ISOS_50R_50W_B5: [],
    YCSB_ISOS_50R_50W_B25: [],
}

load_benchmark_data_into_dict(
    "data/ycsb/50r_50w/isos", ycsb_isos_50r_50w_dfs, ycsb_dir_prefix, ycsb_file_prefix
)

## 95r_5w benchmarks

ycsb_smart_95r_5w_dfs = {
    YCSB_BFT_SMART_LEADER0_95R_5W_3000: [],
    YCSB_BFT_SMART_LEADER1_95R_5W_3000: [],
    YCSB_BFT_SMART_LEADER2_95R_5W_3000: [],
    YCSB_BFT_SMART_LEADER3_95R_5W_3000: [],
}

load_benchmark_data_into_dict(
    "data/ycsb/95r_5w/bft-smart",
    ycsb_smart_95r_5w_dfs,
    ycsb_dir_prefix,
    ycsb_file_prefix,
)

ycsb_isos_95r_5w_dfs = {
    YCSB_ISOS_95R_5W_B5: [],
    YCSB_ISOS_95R_5W_B25: [],
}

load_benchmark_data_into_dict(
    "data/ycsb/95r_5w/isos", ycsb_isos_95r_5w_dfs, ycsb_dir_prefix, ycsb_file_prefix
)

## 100r_0w benchmarks

ycsb_smart_100r_0w_dfs = {
    YCSB_BFT_SMART_LEADER0_100R_0W_3000: [],
    YCSB_BFT_SMART_LEADER1_100R_0W_3000: [],
    YCSB_BFT_SMART_LEADER2_100R_0W_3000: [],
    YCSB_BFT_SMART_LEADER3_100R_0W_3000: [],
}

load_benchmark_data_into_dict(
    "data/ycsb/100r_0w/bft-smart",
    ycsb_smart_100r_0w_dfs,
    ycsb_dir_prefix,
    ycsb_file_prefix,
)

ycsb_isos_100r_0w_dfs = {
    YCSB_ISOS_100R_0W_B5: [],
    YCSB_ISOS_100R_0W_B25: [],
}

load_benchmark_data_into_dict(
    "data/ycsb/100r_0w/isos", ycsb_isos_100r_0w_dfs, ycsb_dir_prefix, ycsb_file_prefix
)

# KVStore

kvstore_smart_dfs = {
    KV_BFT_SMART_LEADER0_1000: [],
    KV_BFT_SMART_LEADER1_1000: [],
    KV_BFT_SMART_LEADER2_1000: [],
    KV_BFT_SMART_LEADER3_1000: [],
}

load_benchmark_data_into_dict(
    "data/kvstore/bft-smart", kvstore_smart_dfs, kvstore_dir_prefix, kvstore_file_prefix
)

kvstore_isos_dfs = {
    KV_ISOS_C0_1000: [],
    KV_ISOS_C2_1000: [],
    KV_ISOS_C5_1000: [],
    KV_ISOS_C100_1000: [],
}

load_benchmark_data_into_dict(
    "data/kvstore/isos", kvstore_isos_dfs, kvstore_dir_prefix, kvstore_file_prefix
)

print("done")

In [ ]:
benchmark_categories = {
    "baseline": [NO_OPT_500],
    "cachemap": [RECORD_CACHE_500],
    "serialization": [CMD_CACHE_500],
    "conflicts": [
        HIGHEST_CONFLICT_500,
    ],
    "depgraph": [
        DEPGRAPH_CACHED_500,
        DEPGRAPH_CONCURRENT_TEST_500_LOCKED,
        DEPGRAPH_CONCURRENT_TEST1_500_PHASER,
        CONCDEP_HIGHESTCONFLICT_500,
    ],
    "execution": [SEQ_EXEC_25K_500, CONC_EXEC_25K_500, CONC_EXEC_25K_50W_500],
    "batching": [BATCH_5_25MS_500, BATCH_5_50MS_500, BATCH_5_500, BATCH_5_150MS_500],
}

category_cmaps = {
    "baseline": plt.get_cmap("Greys"),
    "cachemap": plt.get_cmap("Purples"),
    "serialization": plt.get_cmap("Greens"),
    "conflicts": plt.get_cmap("Oranges"),
    "depgraph": plt.get_cmap("Reds"),
    "execution": plt.get_cmap("Blues"),
    "batching": plt.get_cmap("Purples"),
}

benchmark_colors = {}

for category, items in benchmark_categories.items():
    cmap = category_cmaps[category]
    n = len(items)

    vmin, vmax = 0.4, 0.8

    if n == 1:
        # Just take the mid-point color of the colormap
        values = [0.5 * (vmin + vmax)]
    else:
        # Evenly spaced shades across the colormap
        values = np.linspace(vmin, vmax, n)

    colors = [(cmap(v)) for v in values]

    for item, color in zip(items, colors):
        benchmark_colors[item] = color


In [ ]:
# Summaries
isos_opt_stats = summary_from_df_dict(isos_opt_dfs, 5)
ycsb_smart_50r_50w_stats = summary_from_df_dict(ycsb_smart_50r_50w_dfs, 15)
ycsb_isos_50r_50w_stats = summary_from_df_dict(ycsb_isos_50r_50w_dfs, 15)
ycsb_smart_95r_5w_stats = summary_from_df_dict(ycsb_smart_95r_5w_dfs, 15)
ycsb_isos_95r_5w_stats = summary_from_df_dict(ycsb_isos_95r_5w_dfs, 15)
ycsb_smart_100r_0w_stats = summary_from_df_dict(ycsb_smart_100r_0w_dfs, 15)
ycsb_isos_100r_0w_stats = summary_from_df_dict(ycsb_isos_100r_0w_dfs, 15)
kvstore_smart_stats = summary_from_df_dict(kvstore_smart_dfs, 15)
kvstore_isos_stats = summary_from_df_dict(kvstore_isos_dfs, 15)

In [ ]:
# Sanity check: all data correctly loaded?

averages = []


def add_to_average(stats):
    for summary in stats.values():
        averages.append(summary["avg_latency_ms"].item())


add_to_average(isos_opt_stats)
add_to_average(ycsb_smart_50r_50w_stats)
add_to_average(ycsb_isos_50r_50w_stats)
add_to_average(ycsb_smart_95r_5w_stats)
add_to_average(ycsb_isos_95r_5w_stats)
add_to_average(ycsb_smart_100r_0w_stats)
add_to_average(ycsb_isos_100r_0w_stats)
add_to_average(kvstore_smart_stats)
add_to_average(kvstore_isos_stats)

has_duplicates = len(averages) != len(set(averages))
if len(averages) == 0:
    print("Error: no averages found!")
elif has_duplicates:
    print(
        "Attention: Duplicate average latency found, potentially loaded same data twice?"
    )
else:
    print("No duplicate means found!")


## Visualization

In [ ]:
# Data plotting


def plot_overview(df: pd.DataFrame, benchmark_name: str, output_dir: Path):
    """Plot overview: average latency, throughput and run duration across runs"""
    ops_df = filter_error_df(df)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=double_figsize)

    # Calculate per-run metrics
    run_metrics = []
    for run_num in sorted(ops_df["run_number"].unique()):
        run_df = ops_df[ops_df["run_number"] == run_num]
        duration_s = (
            run_df["absolute_timestamp_ms"].max()
            - run_df["absolute_timestamp_ms"].min()
        ) / 1000

        run_metrics.append(
            {
                "run": run_num,
                "avg_latency_ms": run_df["latency_ms"].mean(),
                "throughput_ops_s": len(run_df) / duration_s if duration_s > 0 else 0,
                "duration_s": duration_s,
            }
        )

    metrics_df = pd.DataFrame(run_metrics)

    # Convert run numbers to string for discrete x-axis
    metrics_df["run_str"] = metrics_df["run"].astype(str)

    # Prepare summary values
    avg_latency = metrics_df["avg_latency_ms"].mean()
    avg_throughput = metrics_df["throughput_ops_s"].mean()

    # Plot 1: Average Latency
    ax1.bar(
        metrics_df["run_str"],
        metrics_df["avg_latency_ms"],
        color="steelblue",
        edgecolor="black",
        linewidth=bar_linewidth,
    )
    ax1.axhline(
        y=avg_latency,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Overall Avg: {avg_latency:.1f} ms",
    )
    ax1.set_xlabel("Run Number", fontsize=xylabel_fontsize)
    ax1.set_ylabel("Average Latency [ms]", fontsize=xylabel_fontsize)
    ax1.set_title("Average Latency per Run", fontsize=title_fontsize)
    ax1.legend()
    ax1.grid(True, axis="y", alpha=0.3)

    # Plot 2: Throughput
    ax2.bar(
        metrics_df["run_str"],
        metrics_df["throughput_ops_s"],
        color="darkgreen",
        edgecolor="black",
        linewidth=bar_linewidth,
    )
    ax2.axhline(
        y=avg_throughput,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Overall Avg: {avg_throughput:.1f} ops/s",
    )
    ax2.set_xlabel("Run Number", fontsize=xylabel_fontsize)
    ax2.set_ylabel("Throughput [ops/s]", fontsize=xylabel_fontsize)
    ax2.set_title("Throughput per Run", fontsize=title_fontsize)
    ax2.legend()
    ax2.grid(True, axis="y", alpha=0.3)

    plt.suptitle(f"{benchmark_name} - Performance Overview", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(output_dir / "overview.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_percentiles_overall(df: pd.DataFrame, benchmark_name: str, output_dir: Path):
    """Plot latency percentiles with bands across all runs"""

    percentiles = [1, 5, 50, 90, 95, 99, 99.9]

    # Calculate percentiles per run
    run_percentiles = []
    for run_num in df["run_number"].unique():
        run_df = df[df["run_number"] == run_num]
        latencies = run_df["latency_ms"]

        run_p = {"run": run_num}
        for p in percentiles:
            run_p[f"p{p}"] = np.percentile(latencies, p)
        run_percentiles.append(run_p)

    perc_df = pd.DataFrame(run_percentiles)

    # Calculate overall statistics
    overall_mean = [perc_df[f"p{p}"].mean() for p in percentiles]
    overall_std = [perc_df[f"p{p}"].std() for p in percentiles]

    fig, ax = plt.subplots(figsize=double_figsize)

    x = np.arange(len(percentiles))

    # Plot mean with error bands
    ax.plot(
        x,
        overall_mean,
        "o-",
        linewidth=2,
        markersize=8,
        color="steelblue",
        label="Mean across runs",
    )

    # Add ±1 std dev band
    ax.fill_between(
        x,
        np.array(overall_mean) - np.array(overall_std),
        np.array(overall_mean) + np.array(overall_std),
        alpha=0.3,
        color="steelblue",
        label="±1 Std Dev",
    )

    # Add individual run data points
    for _, row in perc_df.iterrows():
        run_vals = [row[f"p{p}"] for p in percentiles]
        ax.plot(x, run_vals, "o", alpha=0.4, markersize=5, color="gray")

    ax.set_xticks(x)
    ax.set_xticklabels([f"p{p}" for p in percentiles], fontsize=11)
    ax.set_ylabel("Latency [ms]", fontsize=xylabel_fontsize)
    ax.set_xlabel("Percentile", fontsize=xylabel_fontsize)
    ax.set_title(f"{benchmark_name} - Latency Percentiles", fontsize=title_fontsize)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Add value labels
    for i, (mean_val, std_val) in enumerate(zip(overall_mean, overall_std)):
        ax.text(i, mean_val, f"{mean_val:.1f}", ha="center", va="bottom", fontsize=9)

    plt.tight_layout()
    plt.savefig(output_dir / "percentiles_overall.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_percentiles_by_client(df: pd.DataFrame, benchmark_name: str, output_dir: Path):
    """Plot latency percentiles split by client"""

    percentiles = [1, 5, 50, 90, 95, 99, 99.9]

    fig, ax = plt.subplots(figsize=double_figsize)

    x = np.arange(len(percentiles))
    width = 0.2
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    for i, client_id in enumerate(sorted(df["client_id"].unique())):
        client_df = df[df["client_id"] == client_id]
        latencies = client_df["latency_ms"]

        values = [np.percentile(latencies, p) for p in percentiles]

        offset = width * (i - 1.5)
        ax.bar(
            x + offset,
            values,
            width,
            label=f"Client {region_mapping[str(client_id)]}",
            color=colors[i],
            edgecolor="black",
            linewidth=0.5,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([f"p{p}" for p in percentiles], fontsize=11)
    ax.set_ylabel("Latency [ms]", fontsize=xylabel_fontsize)
    ax.set_xlabel("Percentile", fontsize=xylabel_fontsize)
    ax.set_title(
        f"{benchmark_name} - Latency Percentiles by Client", fontsize=title_fontsize
    )
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_dir / "percentiles_by_client.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_latency_over_time(df: pd.DataFrame, benchmark_name: str, output_dir: Path):
    """Plot average latency over time in 1-second buckets"""

    # Use overall_run_ms which is relative time within each run
    df["time_bucket"] = (df["overall_run_ms"] // 1000).astype(int)

    # Group by run and time bucket
    bucketed = (
        df.groupby(["run_number", "time_bucket"])["latency_ms"].mean().reset_index()
    )

    fig, ax = plt.subplots(figsize=double_figsize)

    # Plot each run
    for run_num in sorted(bucketed["run_number"].unique()):
        run_data = bucketed[bucketed["run_number"] == run_num]
        ax.plot(
            run_data["time_bucket"],
            run_data["latency_ms"],
            alpha=0.3,
            linewidth=1,
            color="gray",
        )

    # Plot overall average across all runs
    overall_avg = bucketed.groupby("time_bucket")["latency_ms"].mean().reset_index()

    ax.plot(
        overall_avg["time_bucket"],
        overall_avg["latency_ms"],
        linewidth=2.5,
        color="steelblue",
        label="Average across all runs",
    )

    ax.set_xlabel("Time [ms]", fontsize=xylabel_fontsize)
    ax.set_ylabel("Average Latency [ms]", fontsize=xylabel_fontsize)
    ax.set_title(
        f"{benchmark_name} - Average Latency Over Time (1000 ms buckets)",
        fontsize=title_fontsize,
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(output_dir / "latency_over_time.png", dpi=300, bbox_inches="tight")
    plt.show()


## ISOS Optimizations

In [ ]:
def plot_benchmark_comparison(
    data_dict,
    column_key,
    unit: str,
    benchmark_names: list[str],
    benchmark_labels: list[str],
    output_dir: Path,
    output_name: str,
    title=None,
):
    # Extract metric values
    values = []
    colors = []
    for name in benchmark_names:
        if name not in data_dict:
            raise KeyError(f"Benchmark '{name}' not found in dictionary")

        df = data_dict[name]
        if column_key not in df.columns:
            raise KeyError(f"Column '{column_key}' not found in benchmark '{name}'")

        values.append(df[column_key].item())
        colors.append(benchmark_colors[name])

    # Plot setup
    plt.figure(figsize=single_figsize)
    plt.bar(
        benchmark_labels,
        values,
        color=colors,
        edgecolor="black",
        linewidth=1,
    )
    plt.ylabel(unit)
    plt.title(title if title else f"Comparison of {column_key}")
    # plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / f"{output_name}.png", dpi=300, bbox_inches="tight")
    plt.show()


def plot_avg_latency_ms_comparison(
    benchmark_names: list[str], benchmark_label: list[str], filename: str, title: str
):
    return plot_benchmark_comparison(
        isos_opt_stats,
        "avg_latency_ms",
        "Average Latency [ms]",
        benchmark_names,
        benchmark_label,
        comparison_output_dir,
        f"{filename}_latency",
        title,
    )


def plot_avg_throughput_comparison(
    benchmark_names: list[str], benchmark_label: list[str], filename: str, title: str
):
    return plot_benchmark_comparison(
        isos_opt_stats,
        "avg_throughput_ops_s",
        "Average Throughput [ops/s]",
        benchmark_names,
        benchmark_label,
        comparison_output_dir,
        f"{filename}_throughput",
        title,
    )


# REQUEST SERIALIZATION CACHE
plot_avg_latency_ms_comparison(
    [NO_OPT_500, CMD_CACHE_500],
    ["Baseline: No optimizations", "Request Serialization Cache\n(RSC) enabled"],
    "request_serialization_cache",
    "Optimization: Request Serialization Cache",
)

# RECORD CACHE MAPS
plot_avg_latency_ms_comparison(
    [CMD_CACHE_500, RECORD_CACHE_500],
    ["Baseline:\nRSC enabled", "CacheMap for Records enabled"],
    "record_cache_maps",
    "Optimization: Record CacheMap",
)


# COMPACT DEPENDENCY SET
plot_avg_latency_ms_comparison(
    [CMD_CACHE_500, HIGHEST_CONFLICT_500],
    [
        "Baseline:\nRSC enabled, Trivial CDS",
        "HighestConflict\nCompact Dependency Set (CDS)",
    ],
    "cds_highest_conflict",
    "Optimization: Compact Dependency Set Strategy",
)

# Dependency Graph Builder: CACHED DEPENDENCY GRAPH, CONCURRENT DEP
plot_avg_latency_ms_comparison(
    [
        CMD_CACHE_500,
        DEPGRAPH_CACHED_500,
        DEPGRAPH_CONCURRENT_TEST_500_LOCKED,
        DEPGRAPH_CONCURRENT_TEST1_500_PHASER,
    ],
    [
        "Baseline:\nRSC enabled,\nTrivial Builder",
        "Cached Builder",
        "Concurrent Builder\n(await/notify)",
        "Concurrent Builder\n(Phaser)",
    ],
    "depgraphbuilder_initial",
    "Optimization: Dependency Graph Builder",
)

# Dependency Graph Builder with highest conflict for compact dependency set enabled
plot_avg_latency_ms_comparison(
    [HIGHEST_CONFLICT_500, CONCDEP_HIGHESTCONFLICT_500],
    [
        "Baseline:\nRSC enabled,\nHighestConflict CDS",
        "RSC enabled,\nHighestConflict CDS,\nConcurrent Builder",
    ],
    "depgraphbuilder_highestconflict",
    "Optimization: Dependency Graph Builder, with HighestConflict CDS",
)

# Parallel Dependency graph execution

plot_avg_latency_ms_comparison(
    [
        CONCDEP_HIGHESTCONFLICT_500,
        SEQ_EXEC_25K_500,
    ],
    [
        "RSC enabled,\nHighestConflict CDS,\nConcurrent Builder,\n0 Workload Iterations",
        "RSC enabled,\nHighestConflict CDS,\nConcurrent Builder,\n25k Workload Iterations",
    ],
    "parallel_execution",
    "Comparison: Normal execution vs execution with artificial workload",
)

plot_avg_latency_ms_comparison(
    [
        SEQ_EXEC_25K_500,
        CONC_EXEC_25K_500,
        CONC_EXEC_25K_50W_500,
    ],
    [
        "Baseline:\nSequential execution",
        "Concurrent execution\n95r / 5w",
        "Concurrent execution\n50r / 50w",
    ],
    "parallel_execution_workload",
    "Optimization: Concurrent Execution in Execution Manager",
)

# Message Batching
plot_avg_latency_ms_comparison(
    [
        CONCDEP_HIGHESTCONFLICT_500,
        BATCH_5_25MS_500,
        BATCH_5_50MS_500,
        BATCH_5_500,
        BATCH_5_150MS_500,
    ],
    [
        "Baseline:\nHighestConflict\nConc. DepGraph",
        "Batch 5,\ntimeout 25 ms",
        "Batch 5,\ntimeout 50 ms",
        "Batch 5,\ntimeout 100 ms",
        "Batch 5,\ntimeout 150 ms",
    ],
    "message_batching",
    "Optimization: Message batching",
)

# for benchmark_name in isos_opt_dfs:
#     output_path = output_base_path / benchmark_name
#     output_path.mkdir(parents=True, exist_ok=True)

#     plot_overview(isos_opt_dfs[benchmark_name], benchmark_name, output_path)
#     #plot_percentiles_overall(test_df, benchmark_name, output_path)
#     #plot_percentiles_by_client(test_df, benchmark_name, output_path)
#     #plot_latency_over_time(test_df, benchmark_name, output_path)

# YCSB Benchmark

In [ ]:
bftSmart = "BFT-SMaRt"


def get_ycsb_summary(key_name: str) -> dict:
    return {
        f"{bftSmart} FRA": [
            ycsb_smart_50r_50w_stats[YCSB_BFT_SMART_LEADER0_50R_50W_3000][
                key_name
            ].item(),
            ycsb_smart_95r_5w_stats[YCSB_BFT_SMART_LEADER0_95R_5W_3000][
                key_name
            ].item(),
            ycsb_smart_100r_0w_stats[YCSB_BFT_SMART_LEADER0_100R_0W_3000][
                key_name
            ].item(),
        ],
        f"{bftSmart} SF": [
            ycsb_smart_50r_50w_stats[YCSB_BFT_SMART_LEADER1_50R_50W_3000][
                key_name
            ].item(),
            ycsb_smart_95r_5w_stats[YCSB_BFT_SMART_LEADER1_95R_5W_3000][
                key_name
            ].item(),
            ycsb_smart_100r_0w_stats[YCSB_BFT_SMART_LEADER1_100R_0W_3000][
                key_name
            ].item(),
        ],
        f"{bftSmart} BAN": [
            ycsb_smart_50r_50w_stats[YCSB_BFT_SMART_LEADER2_50R_50W_3000][
                key_name
            ].item(),
            ycsb_smart_95r_5w_stats[YCSB_BFT_SMART_LEADER2_95R_5W_3000][
                key_name
            ].item(),
            ycsb_smart_100r_0w_stats[YCSB_BFT_SMART_LEADER2_100R_0W_3000][
                key_name
            ].item(),
        ],
        f"{bftSmart} SYD": [
            ycsb_smart_50r_50w_stats[YCSB_BFT_SMART_LEADER3_50R_50W_3000][
                key_name
            ].item(),
            ycsb_smart_95r_5w_stats[YCSB_BFT_SMART_LEADER3_95R_5W_3000][
                key_name
            ].item(),
            ycsb_smart_100r_0w_stats[YCSB_BFT_SMART_LEADER3_100R_0W_3000][
                key_name
            ].item(),
        ],
        "ISOS Batch 5": [
            ycsb_isos_50r_50w_stats[YCSB_ISOS_50R_50W_B5][key_name].item(),
            ycsb_isos_95r_5w_stats[YCSB_ISOS_95R_5W_B5][key_name].item(),
            ycsb_isos_100r_0w_stats[YCSB_ISOS_100R_0W_B5][key_name].item(),
        ],
        "ISOS Batch 25": [
            ycsb_isos_50r_50w_stats[YCSB_ISOS_50R_50W_B25][key_name].item(),
            ycsb_isos_95r_5w_stats[YCSB_ISOS_95R_5W_B25][key_name].item(),
            ycsb_isos_100r_0w_stats[YCSB_ISOS_100R_0W_B25][key_name].item(),
        ],
    }


def print_grouped_ycsb(
    key_name: str, unit: str, title: str, filename: str, percentile: str = "perc_p90"
):
    data = get_ycsb_summary(key_name)

    latency_p99 = None
    if key_name == "avg_latency_ms":
        latency_p99 = get_ycsb_summary(percentile)

    categories = ["50r / 50w", "95r / 5w", "100r / 0w"]

    df = pd.DataFrame(data, index=categories)
    if latency_p99 is not None:
        df_lat = pd.DataFrame(latency_p99, index=categories)
    else:
        df_lat = None

    # Parameters for grouped bar layout
    bar_width = 0.13
    gap = 0.08  # spacing between bft smart and isos
    x = np.arange(len(df.index))  # category positions

    fig, ax = plt.subplots(figsize=single_figsize)

    colors = ["#d73027", "#fc8d59", "#fee090", "#e0c432", "#91bfdb", "#1a9850"]

    # Plot each YCSB workload as bar group
    for i, (col, color) in enumerate(zip(df.columns, colors)):
        shift = (i * bar_width) + (gap if i >= 4 else 0)
        ax.bar(
            x + shift,
            df[col],
            width=bar_width,
            label=col,
            color=color,
            edgecolor="black",
            linewidth=bar_linewidth,
            zorder=2,
        )

    # Add optional percentile for latency
    if df_lat is not None:
        for i, (col, color) in enumerate(zip(df_lat.columns, colors)):
            shift = (i * bar_width) + (gap if i >= 4 else 0) - (bar_width / 2)
            x_center = x + shift
            for xi, thr, lat in zip(x_center, df[col], df_lat[col]):
                ax.errorbar(
                    xi + bar_width / 2,
                    thr,
                    yerr=[np.zeros(1), [lat * 0.7]],  # scaled for visibility
                    fmt="none",
                    ecolor="black",
                    elinewidth=bar_linewidth,
                    capsize=5,
                    capthick=bar_linewidth * 0.9,
                    zorder=1,
                )

    # Center tick positions considering total width (including gap)
    group_width = 6 * bar_width + gap
    ax.set_xticks(x + group_width / 2 - bar_width / 2)
    ax.set_xticklabels(df.index, fontsize=11)

    # TODO: Add vertical lines between categories

    ax.set_ylabel(unit, fontsize=13)
    ax.legend(fontsize=10, ncol=3)
    ax.tick_params(axis="y", labelsize=11)
    ax.set_title(title, fontsize=title_fontsize, pad=15)

    plt.tight_layout()
    plt.savefig(ycsb_output_dir / filename, dpi=300, bbox_inches="tight")
    plt.show()


print_grouped_ycsb(
    "avg_latency_ms", "Latency [ms]", "YCSB Average Latency, p90", "ycsb_latency_p90"
)
print_grouped_ycsb(
    "avg_latency_ms",
    "Latency [ms]",
    "YCSB Average Latency, p99",
    "ycsb_latency_p99",
    "perc_p99",
)
print_grouped_ycsb(
    "avg_throughput_ops_s",
    "Throughput [ops/s]",
    "YCSB Average Throughput",
    "ycsb_throughput",
)

# KVStore Latency / Conflicts

In [ ]:
# Load summary data for KVStore into single dataframe


def split_by_client(df):
    client_ids = sorted(df["client_id"].unique())
    return [df[df["client_id"] == i] for i in client_ids]


def get_kvstore_dfs() -> dict:
    return {
        f"{bftSmart} Leader FRA": split_by_client(
            kvstore_smart_dfs[KV_BFT_SMART_LEADER0_1000]
        ),
        f"{bftSmart} Leader SF": split_by_client(
            kvstore_smart_dfs[KV_BFT_SMART_LEADER1_1000]
        ),
        f"{bftSmart} Leader BAN": split_by_client(
            kvstore_smart_dfs[KV_BFT_SMART_LEADER2_1000]
        ),
        f"{bftSmart} Leader SYD": split_by_client(
            kvstore_smart_dfs[KV_BFT_SMART_LEADER3_1000]
        ),
        "ISOS 0%": split_by_client(kvstore_isos_dfs[KV_ISOS_C0_1000]),
        "ISOS 2%": split_by_client(kvstore_isos_dfs[KV_ISOS_C2_1000]),
        "ISOS 5%": split_by_client(kvstore_isos_dfs[KV_ISOS_C5_1000]),
        "ISOS 100%": split_by_client(kvstore_isos_dfs[KV_ISOS_C100_1000]),
    }


In [ ]:
# Sanity check
averages = []

summary_data = summary_from_df_array_dict(get_kvstore_dfs(), 15)

for benchmark in summary_data:
    for client_summary in summary_data[benchmark]:
        averages.append(client_summary["avg_latency_ms"].item())


has_duplicates = len(averages) != len(set(averages))
if len(averages) == 0:
    print("Error: No averages found!")
elif has_duplicates:
    print(
        "Attention: Duplicate average latency found, potentially loaded same data twice?"
    )
else:
    print(f"No duplicate means found from {len(averages)} summaries!")


In [ ]:
# Plot in groups of client latencies for KVStore benchmark


def print_grouped_kvstore(
    unit: str, title: str, filename: str, percentile: str = "perc_p90"
):
    summary_data = summary_from_df_array_dict(get_kvstore_dfs(), 15)

    latency_data = {}
    p99_data = {}

    for benchmark, client_summaries in summary_data.items():
        latency_data[benchmark] = [
            client["avg_latency_ms"].item() for client in client_summaries
        ]
        p99_data[benchmark] = [client[percentile].item() for client in client_summaries]

    categories = ["FRA Clients", "SF Clients", "BAN Clients", "SYD Clients"]

    df = pd.DataFrame(latency_data, index=categories)
    df_lat = pd.DataFrame(p99_data, index=categories)

    # Parameters for grouped bar layout
    bar_width = 0.1
    gap = 0.05  # spacing between bft smart and isos
    x = np.arange(len(df.index))  # category positions

    fig, ax = plt.subplots(figsize=single_figsize)

    colors = [
        "#d73027",
        "#fc8d59",
        "#fee090",
        "#e0c432",
        "#006400",  # ISOS 0%
        "#228B22",  # ISOS 2%
        "#32CD32",  # ISOS 5%
        "#ADFF2F",  # ISOS 100%
    ]

    # Plot each YCSB workload as bar group
    for i, (col, color) in enumerate(zip(df.columns, colors)):
        shift = (i * bar_width) + (gap if i >= 4 else 0)
        ax.bar(
            x + shift,
            df[col],
            width=bar_width,
            label=col,
            color=color,
            edgecolor="black",
            linewidth=bar_linewidth,
            zorder=2,
        )

    for i, (col, color) in enumerate(zip(df_lat.columns, colors)):
        shift = (i * bar_width) + (gap if i >= 4 else 0) - (bar_width / 2)
        x_center = x + shift
        for xi, thr, lat in zip(x_center, df[col], df_lat[col]):
            ax.errorbar(
                xi + bar_width / 2,
                thr,
                yerr=[[0], [lat * 0.7]],  # scaled for visibility
                fmt="none",
                ecolor="black",
                elinewidth=bar_linewidth,
                capsize=3,
                capthick=bar_linewidth * 0.9,
                zorder=1,
            )

    # Center tick positions considering total width (including gap)
    group_width = 6 * bar_width + gap
    ax.set_xticks(x + group_width / 2 - bar_width / 2)
    ax.set_xticklabels(df.index, fontsize=11)

    # TODO: Add vertical lines between categories

    ax.set_ylabel(unit, fontsize=13)
    ax.legend(fontsize=10, ncol=3, bbox_to_anchor=(0.5, -0.1), loc="upper center")
    ax.tick_params(axis="y", labelsize=11)
    ax.set_title(title, fontsize=title_fontsize, pad=15)

    plt.ylim(0, 2000)
    plt.tight_layout()
    plt.savefig(kvstore_output_dir / filename, dpi=300, bbox_inches="tight")
    plt.show()


print_grouped_kvstore(
    "Response time [ms]",
    "KVStore Average Latency, p90",
    "kvstore_latency_p90",
    "perc_p90",
)
print_grouped_kvstore(
    "Response time [ms]",
    "KVStore Average Latency, p99",
    "kvstore_latency_p99",
    "perc_p99",
)

# Visualization of all runs

In [ ]:
print("ISOS Optimizations")
for benchmark_name in isos_opt_dfs:
    output_path = detailed_output_dir / benchmark_name
    output_path.mkdir(parents=True, exist_ok=True)

    plot_overview(isos_opt_dfs[benchmark_name], benchmark_name, output_path)
    # plot_percentiles_overall(test_df, benchmark_name, output_path)
    # plot_percentiles_by_client(test_df, benchmark_name, output_path)
    # plot_latency_over_time(test_df, benchmark_name, output_path)

print("YCSB Benchmark")

ycsb_dict = {
    **ycsb_smart_95r_5w_dfs,
    **ycsb_isos_95r_5w_dfs,
    **ycsb_smart_50r_50w_dfs,
    **ycsb_isos_50r_50w_dfs,
    **ycsb_smart_100r_0w_dfs,
    **ycsb_isos_100r_0w_dfs,
}


for benchmark_name in ycsb_dict:
    output_path = detailed_output_dir / benchmark_name
    output_path.mkdir(parents=True, exist_ok=True)

    plot_overview(ycsb_dict[benchmark_name], benchmark_name, output_path)


print("KVStore Benchmark")
for benchmark_name in kvstore_isos_dfs:
    output_path = detailed_output_dir / benchmark_name
    output_path.mkdir(parents=True, exist_ok=True)

    plot_overview(kvstore_isos_dfs[benchmark_name], benchmark_name, output_path)

for benchmark_name in kvstore_smart_dfs:
    output_path = detailed_output_dir / benchmark_name
    output_path.mkdir(parents=True, exist_ok=True)

    plot_overview(kvstore_smart_dfs[benchmark_name], benchmark_name, output_path)